# USD/CAD — Your Starter Agent

This notebook implements the starter agent for the USD/CAD forecasting use case.

**Target:** `usdcad_logret_{h}b = log(DEXCAUS[t+h] / DEXCAUS[t])`.

Because FRED `DEXCAUS` is Canadian dollars per U.S. dollar, a positive target return means USD/CAD rises (USD strengthens relative to CAD); a negative return means CAD strengthens relative to USD.

The agent supports cutoff-aware web research, optional code diagnostics, and the leak-safe FRED covariate panel already implemented in this use case.

In [ ]:
import warnings
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parents[1]
load_dotenv(ROOT / ".env", override=False)

AGENT_MODEL = "gemini-3.1-flash-lite-preview"
# AGENT_MODEL = "gemini-3.5-flash"
RUN_AGENT = False

from usdcad_forecasting import DEFAULT_COVARIATE_SERIES_IDS
from usdcad_forecasting.starter_agent import (
    build_starter_agent_config,
    build_starter_agent_predictor,
)

print("RUN_AGENT =", RUN_AGENT, "| model =", AGENT_MODEL)
print("Default covariates:", DEFAULT_COVARIATE_SERIES_IDS)

---
## 1. Meet your agent

Search is enabled by default; code execution is optional. The forecasting skill is always loaded.

In [ ]:
config = build_starter_agent_config(
    model=AGENT_MODEL,
    enable_search=True,
    enable_code_exec=False,
)

print("Agent:", config.name)
print("Search enabled:", config.context_retrieval.enabled)
print("Code-exec enabled:", config.code_execution.enabled)
print("Skills loaded:", [p.name for p in config.skills_dirs])
print("\nSystem instruction:\n")
print(config.instruction)

---
## 2. Talk to it — USD/CAD market analysis

Use this mode for exploratory FX questions. It does not require the structured forecast output schema.

In [ ]:
from aieng.forecasting.methods.agentic import build_adk_agent
from aieng.forecasting.methods.agentic.adk_runner import AdkTextRunner, AdkTextRunnerConfig

QUESTION = (
    "What are the main USD/CAD drivers right now? Discuss Bank of Canada and "
    "Federal Reserve policy, US-Canada rate differentials, oil, and Canadian/US "
    "macro data. Keep it concise."
)

if RUN_AGENT:
    chat_agent = build_adk_agent(config)
    runner = AdkTextRunner(
        chat_agent,
        config=AdkTextRunnerConfig(app_name="usdcad_starter_chat"),
    )
    reply = await runner.run_text_async(QUESTION)
    print(reply)
else:
    print("RUN_AGENT is False — set it to True to talk to the agent.")

---
## 3. Score one resolved forecast

The cell below forecasts a single resolved origin. Start with 5 business days, then repeat for 1 and 21.

In [ ]:
from datetime import datetime, timezone
from aieng.forecasting.evaluation.task import ForecastingTask
from usdcad_forecasting import (
    build_usdcad_multivariate_service,
    usdcad_logret_series_id,
)

HORIZON = 5
COVARIATES = DEFAULT_COVARIATE_SERIES_IDS

if RUN_AGENT:
    svc = build_usdcad_multivariate_service(
        covariate_series_ids=COVARIATES,
    )
    now = datetime.now(tz=timezone.utc).replace(tzinfo=None)
    tgt = usdcad_logret_series_id(HORIZON)
    full = svc.get_series(tgt, as_of=now)
    full["timestamp"] = pd.to_datetime(full["timestamp"])
    last_date = full["timestamp"].iloc[-1]

    # Pick the most recent origin with a fully resolved H-day target.
    AS_OF = last_date - pd.offsets.BDay(HORIZON + 1)

    task = ForecastingTask(
        task_id=f"usdcad_logret_{HORIZON}b",
        target_series_id=tgt,
        horizons=[HORIZON],
        frequency="B",
        description=f"USD/CAD cumulative log return, {HORIZON} business days ahead.",
    )

    ctx = svc.context(as_of=AS_OF)
    pred = build_starter_agent_predictor(
        config,
        covariate_series_ids=COVARIATES,
    ).predict(task, ctx)[0]

    rows = full[full["timestamp"] >= AS_OF + pd.offsets.BDay(HORIZON)]
    actual = float(rows["value"].iloc[0]) if not rows.empty else None

    fc = pred.payload
    lo = fc.quantiles[0.10]
    hi = fc.quantiles[0.90]

    print(
        f"Origin as_of={AS_OF.date()} | horizon={HORIZON}b | "
        f"latest target data={last_date.date()}\n"
    )
    print(
        f"  agent point  : {fc.point_forecast:+.6f} "
        f"({fc.point_forecast * 100:+.3f}%)"
    )
    print(f"  agent 80% CI : [{lo:+.6f}, {hi:+.6f}]")

    if actual is None:
        print("  actual       : N/A")
    else:
        in_band = "yes ✓" if lo <= actual <= hi else "no ✗"
        print(
            f"  actual       : {actual:+.6f} "
            f"({actual * 100:+.3f}%) | in 80% band? {in_band}"
        )

    if pred.metadata.get("rationale"):
        print("\nRationale:", pred.metadata["rationale"][:500])
else:
    print("RUN_AGENT is False — set it to True to score a live forecast.")

---
## 4. Try the three horizons

Use `HORIZON = 1`, `5`, or `21`. These map to `usdcad_logret_1b`, `usdcad_logret_5b`, and `usdcad_logret_21b`.

Recommended progression:
1. Target-only agent.
2. Agent + default FRED covariates.
3. Agent + cutoff-aware search.
4. Agent + search + code diagnostics.
5. Compare configurations across the multivariate backtest.

## 5. Covariates

The current default panel contains lagged USD/CAD return, US effective fed funds, US 2Y and 10Y yields, the 2Y–10Y spread, CPI growth, unemployment, and WTI return. These are constructed by the existing data module with its anti-leakage policy.